In [2]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
from functools import partial
from jax import flatten_util


# ==============================================================================
# 1. 全局参数 & H₂ 分子定义
# ==============================================================================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

# ==============================================================================
# 2. 神经网络 Ansatz 
# ==============================================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

# ==============================================================================
# 4. 初始化模型、采样器、优化器
# ==============================================================================
model = SingleStateAnsatz(4,12, rngs=nnx.Rngs(21))
# 采样器
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=100, sweep_size=32)
optimizer = nk.optimizer.Sgd(learning_rate=0.1)

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


In [3]:
# ===================== 4. 包装模型为 machine 函数 =====================
def create_machine(model: nnx.Module):
    """将 Flax NNX 模型包装为 NetKet 风格的 machine 函数"""
    graphdef, state = nnx.split(model)
    
    @jax.jit
    def machine(params, sigma):
        m = nnx.merge(graphdef, params)
        return m(sigma)
    
    return machine, graphdef, state

# ===================== 5. 纯 JAX 实现的 force-based 梯度计算 =====================
@partial(jax.jit, static_argnames=("machine",))
def compute_local_energies(machine, params, sigma):
    """
    计算局部能量 E_loc(σ) = Σ_η H(σ→η) ψ(η)/ψ(σ)
    
    这对应 NetKet 的 local_value_kernel
    """
    eta, H_eta = ha.get_conn_padded(sigma)
    logpsi_sigma = machine(params, sigma)
    logpsi_eta = machine(params, eta)
    logpsi_sigma = jnp.expand_dims(logpsi_sigma, -1)
    return jnp.sum(H_eta * jnp.exp(logpsi_eta - logpsi_sigma), axis=-1)


def statistics(x):
    """计算样本统计量"""
    mean = jnp.mean(x)
    var = jnp.var(x)
    return mean, jnp.sqrt(var / x.shape[0])


@partial(jax.jit, static_argnames=("machine",))
def forces_expect_hermitian(machine, params, sigma):
    """
    核心：复刻 NetKet 的 forces_expect_hermitian 函数
    
    使用 force-based 梯度计算：
    ∇⟨E⟩ = ⟨(E_loc - ⟨E⟩) ∇log ψ⟩
    
    关键：对于复数值网络，使用 holomorphic=True
    """
    # 1. 计算局部能量
    O_loc = compute_local_energies(machine, params, sigma)
    
    # 2. 统计能量均值
    O_mean, O_std = statistics(O_loc)
    
    # 3. 中心化局部能量
    O_centered = O_loc - O_mean
    
    # 4. 计算 ∇log ψ 对每个样本
    # 使用 jax.grad 计算复数梯度（holomorphic=True）
    def log_psi_single(p, s):
        return machine(p, s)
    
    def compute_grad_for_sample(s):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=True)(params)
    
    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)
    
    # 5. 计算 force-based 梯度
    # grad = ⟨(E_loc - E_mean) ∇log ψ⟩ = (1/N) Σ (E_loc[i] - E_mean) ∇log ψ(σ[i])
    # grad_matrix 已经是 PyTree 结构，每个元素的形状是 (n_samples, ...)
    # 关键修复：O_centered 形状为 (n_samples,)，需要正确广播到梯度张量的每个维度
    # 使用 reshape 将 O_centered 变为 (n_samples, 1, 1, ..., 1) 以匹配梯度张量
    def weight_and_mean(grad_component):
        # grad_component 形状：(n_samples, d1, d2, ...)
        # O_centered 形状：(n_samples,)
        # 需要广播相乘后沿 axis=0 求平均
        weights = O_centered.reshape((O_centered.shape[0],) + (1,) * (grad_component.ndim - 1))
        return jnp.mean(weights * jnp.conj(grad_component), axis=0)
    
    grad = jax.tree_util.tree_map(weight_and_mean, grad_matrix)
    
    return O_mean, O_std, grad


#@partial(jax.jit, static_argnames=("machine",))
def compute_qgt(machine, params, sigma, diag_shift=0.1):
    """
    计算量子几何张量（QGT）/ F 矩阵
    
    QGT 定义：
    S_ij = ⟨∂_i log ψ* ∂_j log ψ⟩ - ⟨∂_i log ψ*⟩⟨∂_j log ψ⟩
    
    这就是 NetKet SR 的核心
    
    参数：
    - machine: 波函数机器
    - params: 网络参数
    - sigma: 样本 (n_samples, n_orbitals)
    - diag_shift: 对角线正则化参数 λ
    
    返回：
    - qgt_reg: 正则化后的 QGT 矩阵 (n_params, n_params)
    - unravel_fn: 用于将展平的向量恢复为 PyTree 结构的函数
    """
    n_samples = sigma.shape[0]
    
    # 步骤 1: 计算每个样本的 ∇log ψ
    def log_psi_single(p, s):
        return machine(p, s)
    
    def compute_grad_for_sample(s):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=True)(params)
    
    # grad_matrix 是 PyTree，每个元素形状为 (n_samples, ...)
    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)
    
    # 步骤 2: 将 PyTree 展平为矩阵 (n_samples, n_params)
    grad_flat, unravel_fn = flatten_util.ravel_pytree(grad_matrix)
    grad_flat = grad_flat.reshape(n_samples, -1)
    
    # 步骤 3: 中心化（减去均值）
    # 这对应 QGT 定义中的第二项：- ⟨∂_i log ψ*⟩⟨∂_j log ψ⟩
    grad_mean = jnp.mean(grad_flat, axis=0, keepdims=True)  # (1, n_params)
    grad_centered = grad_flat - grad_mean  # (n_samples, n_params)
    
    # 步骤 4: 计算 QGT = (1/N) * Σ ∇log ψ* ∇log ψ^T
    # 注意：对于复数，需要使用共轭
    qgt = (1.0 / n_samples) * jnp.conj(grad_centered).T @ grad_centered
    
    # 步骤 5: 添加正则化
    qgt_reg = qgt + diag_shift * jnp.eye(qgt.shape[0])
    
    return qgt_reg, unravel_fn

In [6]:
@partial(jax.jit, static_argnames=("machine",))
def forces_expect_with_penalty(machine, params_es, params_gs, sigma_es,
                               lambda_penalty, hamiltonian):
    """
    - 只使用激发态采样 sigma_es
    - 基态参数固定，不采样
    - 标准正交惩罚 λ |⟨Ψgs|Ψes⟩|²
    """
    # ===================== 1. 能量项 =====================
    E_loc = compute_local_energies(machine, params_es, sigma_es, hamiltonian)
    E_mean, E_std = statistics(E_loc)
    E_loc_centered = E_loc - E_mean

    # ===================== 2. 重叠度（仅用 es 样本） =====================
    # 正确公式：⟨Ψ0|Ψ1⟩ ≈ mean( ψ0/ψ1 )，σ ~ ψ1
    psi_es = machine(params_es, sigma_es)
    psi_gs = machine(params_gs, sigma_es)
    psi_ratio = jnp.exp(psi_gs - psi_es)
    overlap = jnp.mean(psi_ratio)
    overlap_sq = jnp.abs(overlap) ** 2

    # ===================== 3. 总损失 =====================
    penalty = lambda_penalty * overlap_sq
    total_loss = E_mean.real + penalty

    # ===================== 4. 能量梯度 =====================
    def log_psi_single(p, s):
        return machine(p, s)

    grad_psi = jax.vmap(jax.grad(lambda p: log_psi_single(p, s), holomorphic=True),
                        in_axes=(None, 0))(params_es, sigma_es)

    def weight(g):
        w = E_loc_centered.reshape(-1, *((1,) * (g.ndim - 1)))
        return jnp.mean(w * jnp.conj(g), axis=0)

    grad_E = jax.tree_map(weight, grad_psi)

    # ===================== 5. 惩罚项梯度 =====================
    def penalty_loss(p):
        psi_e = machine(p, sigma_es)
        ratio = jnp.exp(psi_gs - psi_e)
        ov = jnp.mean(ratio)
        return lambda_penalty * jnp.abs(ov) ** 2

    grad_penalty = jax.grad(penalty_loss, holomorphic=True)(params_es)

    # ===================== 6. 总梯度 =====================
    total_grad = jax.tree_map(lambda ge, gp: ge + gp, grad_E, grad_penalty)

    return total_loss, E_mean, E_std, total_grad, overlap_sq


def compute_qgt(machine, params, sigma, diag_shift=0.1):
    """计算量子几何张量（QGT）/ 自然梯度预 conditioner"""
    n_samples = sigma.shape[0]
    
    def log_psi_single(p, s):
        return machine(p, s)
    
    def compute_grad_for_sample(s):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=True)(params)
    
    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)
    
    grad_flat, unravel_fn = flatten_util.ravel_pytree(grad_matrix)
    grad_flat = grad_flat.reshape(n_samples, -1)
    
    grad_mean = jnp.mean(grad_flat, axis=0, keepdims=True)
    grad_centered = grad_flat - grad_mean
    
    qgt = (1.0 / n_samples) * jnp.conj(grad_centered).T @ grad_centered
    qgt_reg = qgt + diag_shift * jnp.eye(qgt.shape[0])
    
    return qgt_reg, unravel_fn

In [72]:
import orbax.checkpoint as ocp
from pathlib import Path

# 1. 获取当前工作目录的绝对路径
current_dir = Path.cwd().parent  # 或者用 Path(__file__).parent 如果在脚本中
ckpt_dir = current_dir / "暂存态"
# 2. 使用 PyTreeCheckpointer 而不是 StandardCheckpointer
checkpointer = ocp.PyTreeCheckpointer()

# 3. 保存模型（使用绝对路径）
graphdef, state = nnx.split(model)
save_path = str(ckpt_dir / "ground_state_24")  # 转为字符串
restore_state = checkpointer.restore(save_path, state)

In [76]:
ground_state_model = nnx.merge(graphdef,restore_state)
ground_state_model(hi.all_states())

Array([-0.10440015-0.46132108j,  0.26115219-0.68986232j,
       -1.49419227-0.15853739j, -0.48179849-0.13581633j],      dtype=complex128)

In [75]:
state1 = hi.all_states()[0]
state2 = hi.all_states()[1]
state3 = hi.all_states()[2]
state4 = hi.all_states()[3]

E_1 = state1@ha.to_dense()@state1.conj().T
E_2 = state2@ha.to_dense()@state2.conj().T
E_3 = state3@ha.to_dense()@state3.conj().T
E_4 = state4@ha.to_dense()@state4.conj().T

E = [E_1,E_2,E_3,E_4]

logpsi = ground_state_model(hi.all_states())
psi = jnp.exp(logpsi)
born_prob = psi**2
born_prob /= jnp.sum(born_prob)
gs_e = 0
for i in range(4):
    gs_e += born_prob[i]*E[i]

gs_e

Array(-1.08786711-0.05319251j, dtype=complex128)